# Causal DML - FX Momentum

Double machine learning estimates the effect associated with a specified momentum treatment after
flexible adjustment for the configured volatility and price-state variables. It is a causal
estimand, not a predictive model configuration, so its result remains separate from prediction
catalogs and backtest selection.

The shared request verifies that the treatment and every confounder exist, excludes outcomes whose
horizon reaches the holdout, keeps complete timestamp panels when applying a sample cap, cross-fits
nuisance models with an embargo, and records the block-placebo design in the causal identity.

**Learning objectives**

- Inspect the treatment, confounders, timing, and refutation design before estimation.
- Run the same causal request used by direct Python callers.
- Keep causal evidence separate from predictive-family comparison and selection.

**Book reference**: Chapter 15, Section 15.4

**Prerequisites**: `02_labels`, `03_financial_features`, and `04_model_based_features`.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 因果雙重機器學習——外匯動能

雙重機器學習在彈性調整預先設定的波動率與價格狀態變數後，估計與指定動能處置相關的效果。這是一個因果估計目標，不是預測模型配置，因此其結果與預測目錄及回測選擇分開處理。

共用請求會驗證處置變數與每個混雜變數確實存在，排除結果觀測期延伸至保留樣本的資料，在套用樣本數上限時保留完整時間戳記的面板資料，以禁運期進行干擾模型的交叉擬合，並將區塊安慰劑設計記錄在因果識別資訊中。

**學習目標**

- 在估計前檢視處置、混雜變數、時間安排與反駁檢驗設計。
- 執行與直接使用 Python 的呼叫者相同的因果請求。
- 將因果證據與預測模型系列的比較及選擇分開。

**書籍參考**：第 15 章第 15.4 節

**先備條件**：`02_labels`、`03_financial_features` 與 `04_model_based_features`。
<!-- ML4T_ZH_END c0000-p000 -->


## The question this asks, and why it is not the question the rest of the case study asks

Every other modelling notebook here asks a predictive question: given what is observable now,
what is the best guess at next period's return? A model that answers it well is useful even if
every variable in it is a proxy for something else, because prediction does not require knowing
why a relationship holds - only that it keeps holding.

This notebook asks a different question. It asks what would happen to the outcome **if the
treatment were different and nothing else changed**. That is a claim about intervention rather
than association, and it is not answered by a better fit. A model can predict returns from
momentum superbly because both are driven by a third thing, and be completely wrong about what
changing momentum would do.

The distinction matters for what may be done with the answer. A predictive result earns its
place by out-of-sample performance and is selected against other configurations on validation
Sharpe. A causal estimate is not a candidate for that comparison at all - it does not compete
with the gradient boosting configurations, it is not eligible for a backtest, and it never
enters the selection funnel. It answers a question about the market's structure that the
chapter's narrative rests on, and the value of getting it right is that the narrative is true.

### What double machine learning does about confounding

The obstacle is that momentum is not assigned at random. Periods of strong momentum differ
systematically from periods of weak momentum in volatility, in trend state, in liquidity - and
those differences also move returns. Regressing the outcome on the treatment attributes all of
that to the treatment.


<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 本分析要回答的問題，以及為何不同於案例研究其他部分的問題

這裡其他每份建模 Notebook 都在問預測問題：給定目前可觀測的資訊，對下一期報酬率的最佳猜測是什麼？能妥善回答此問題的模型，即使其中每個變數都只是其他因素的代理變數，仍然有用；因為預測無須知道關係為何成立，只須知道它會持續成立。

本 Notebook 問的是另一個問題：**如果處置不同，而其他條件都不變**，結果會如何？這是關於介入而非相關性的主張，無法靠更好的擬合來回答。模型可能因動能與報酬率都受第三個因素驅動，而能出色地用動能預測報酬率；但對改變動能的效果，卻可能完全判斷錯誤。

這項區別會影響結果的用途。預測結果憑樣本外表現取得地位，並以驗證期夏普比率與其他配置競爭選擇。因果估計根本不是這項比較的候選者——它不與梯度提升配置競爭、不具回測資格，也絕不進入選擇流程。它回答的是本章敘事所依據的市場結構問題；得到正確答案的價值，在於敘事確實為真。

### 雙重機器學習如何處理混雜因素

難題在於動能並非隨機分配。高動能時期與低動能時期，在波動率、趨勢狀態、流動性方面有系統性差異，而這些差異也會影響報酬率。若直接將結果對處置變數迴歸，就會把所有這些影響都歸因於處置。
<!-- ML4T_ZH_END c0001-p000 -->

The classical fix is to include the confounders as controls, which works only if their
relationship to the outcome is the shape the model assumes. Double machine learning removes that
requirement in two steps. First it predicts the outcome from the confounders alone, and the
treatment from the confounders alone, using flexible models that need not be linear in anything.
Then it estimates the effect from what is left over in each - the parts of the outcome and the
treatment that the confounders could not explain. Whatever the confounders drive has been taken
out of both sides before the effect is estimated.

The nuisance predictions are **cross-fitted**: the model that residualizes an observation is
never fitted on that observation. Without that, a flexible model overfits its own training rows,
the residuals are too small there, and the effect estimate inherits a bias that grows with how
flexible the nuisance models were. The embargo extends the same idea across time, because a
financial panel's neighbouring rows are not independent the way a cross-section's are.

### What the refutation is for

A causal estimate cannot be validated the way a prediction can, because the counterfactual is
never observed - there is no held-out truth to score against. What can be done is to run the
same estimator against data where the answer is known in advance to be nothing, and check that
nothing is what comes back.

That is the block placebo. The treatment is shuffled so that any genuine relationship to the
outcome is destroyed, the whole estimation is re-run, and the effect it reports is recorded. Do
that many times and the result is a distribution of effects under the null hypothesis that the
treatment does nothing, which is what the reported p-value is computed against.


<!-- ML4T_ZH_BEGIN c0001-p001 -->
傳統修正方式是將混雜變數納入控制變數，但只有在它們與結果之間的關係符合模型假設的形式時才有效。雙重機器學習分兩步移除這項要求。首先，使用不必對任何變數呈線性關係的彈性模型，分別僅依混雜變數預測結果，以及僅依混雜變數預測處置。接著，從兩者各自剩餘的部分——混雜變數無法解釋的結果與處置部分——估計效果。凡是由混雜變數驅動的部分，都已在估計效果前從兩邊移除。

干擾模型的預測採用**交叉擬合**：用來對某筆觀測值取殘差的模型，絕不會以該筆觀測值擬合。若非如此，彈性模型會過度擬合自己的訓練資料列，使那些資料列的殘差過小，效果估計也會承受隨干擾模型彈性增加而加重的偏差。禁運期將同一概念延伸到時間維度，因為金融面板中相鄰的資料列，不像橫斷面觀測值那樣獨立。

### 反駁檢驗的用途

因果估計無法像預測那樣驗證，因為反事實永遠無法觀測——沒有保留的真值可用來評分。可以做的是，對事先已知答案應為零的資料執行相同估計器，檢查回傳結果是否確實為零。

這就是區塊安慰劑檢驗。將處置變數打亂，破壞它與結果之間任何真實的關係，重新執行完整估計，並記錄回報的效果。重複許多次，就會得到在「處置沒有作用」的虛無假設下的效果分布，報告的 p 值即以此分布計算。
<!-- ML4T_ZH_END c0001-p001 -->

The shuffling is done in **blocks** rather than row by row, and the block size is part of the
causal identity for a reason worth stating: a row-by-row shuffle would destroy the series'
autocorrelation along with its relationship to the outcome, and an estimator tested against
implausibly clean data will pass a test that means nothing. The blocks are sized to preserve the
dependence that makes the real series hard, so the placebo is a fair opponent.

<!-- ML4T_ZH_BEGIN c0001-p002 -->
打亂是以**區塊**而非逐列進行，區塊大小也有充分理由被納入因果識別資訊：逐列打亂會在破壞與結果之間關係的同時，也破壞序列的自相關；若拿不合理地乾淨的資料測試估計器，即使通過，也毫無意義。區塊大小設定成能保留使真實序列難以分析的相依性，讓安慰劑成為公平的對照。
<!-- ML4T_ZH_END c0001-p002 -->


In [1]:
"""Estimate and register the configured FX causal DML specification."""

import polars as pl
import yaml

from case_studies.research import ExecutionTier, causal_supersedes, open_study
from utils.paths import get_case_study_dir
from utils.reproducibility import set_global_seeds

In [2]:
CASE_STUDY_ID = "fx_pairs"
PRIMARY_LABEL = ""
MAX_SYMBOLS = 0
CV_FOLDS = 0
MAX_SAMPLES = 0
N_PLACEBO = 0
FORCE_RETRAIN = False
# The tier is a parameter, not something inferred from whether a reduction happens to be set.
# Inferring it meant a reduced run still opened the case study's own artifacts in place, which
# is the production path. WORKSPACE is the other half: a preview has nowhere else to write.
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
# The three identities this run retires, one per label. `CausalResult.one` resolves a label to
# exactly one canonical identity, so the registry refuses a second without being told which one
# it replaces.
#
# These are the fits that *corrected* the placebo block: sizing it by the treatment window
# rather than the label buffer, after the blocks-of-1/5/21 fits of 2026-08-21 had measured a
# placebo that already destroyed the dependence it was meant to preserve. That correction is
# two generations back and is not what moves the hash here.
#
# A reader's clone holds no causal rows at all, so `causal_supersedes` withholds these against
# a registry that does not have them and the reader registers a first identity. That resolution
# has to happen against the registry rather than by leaving the default empty:
# `run-production-notebook.sh` executes with no parameter overrides, so a value supplied only at
# run time could never be stamped.
# Retired by this run: the block-permutation refutation now compares the HAC t-statistic
# rather than the raw effect, so CAUSAL_RUNNER_VERSION moved and every causal identity with
# it. The rows named here hold a p-value computed on the shrunken placebo effects; this run
# supersedes them rather than correcting them, because the statistic is different, not the
# arithmetic. Read out of each registry's current canonical identity per label, 2026-09-10.
SUPERSEDES_CAUSAL: str = (
    '{"fwd_ret_1d": "22bbd3a1e04c", "fwd_ret_5d": "66e5787d22e8", "fwd_ret_21d": "633664501fda"}'
)

## Resolve the estimand and analysis population

Nonzero fold, sample, symbol, or placebo limits declare a preview. They are recorded in identity
and excluded from canonical evidence.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 解析估計目標與分析樣本集合

非零的驗證折、樣本、交易標的或安慰劑次數上限，代表預覽執行。這些限制會記錄在識別資訊中，並排除於標準因果證據之外。
<!-- ML4T_ZH_END c0004-p000 -->


In [3]:
case_dir = get_case_study_dir(CASE_STUDY_ID)
setup = yaml.safe_load((case_dir / "config" / "setup.yaml").read_text())
# The labels this case study declares, and the labels this run fits. They are the same for a
# canonical run and differ whenever PRIMARY_LABEL narrows it.
#
# The distinction decides how `SUPERSEDES_CAUSAL` is read. That declaration names one retired
# identity per declared label, and it is validated against a label set to catch a typo - a hash
# attached to a label that does not exist would retire nothing and fail at registration, after
# the fit is paid for. Validating it against the RUN's labels instead made every narrowed run
# fail that check, because naming three labels while fitting one looked identical to a typo.
# Validating against the declared set keeps the typo caught and lets a narrowed run read the
# entry for the label it is actually fitting.
DECLARED_LABELS = [setup["labels"]["primary"], *setup["labels"].get("variants", [])]
labels = [PRIMARY_LABEL] if PRIMARY_LABEL else list(DECLARED_LABELS)

if FORCE_RETRAIN:
    raise ValueError("an identical complete causal request is reused; change the request to refit")

REDUCTION_PARAMETERS = {
    "max_symbols": MAX_SYMBOLS or None,
    "n_folds": CV_FOLDS or None,
    "max_samples": MAX_SAMPLES or None,
    "n_placebo": N_PLACEBO or None,
}
reductions = {key: value for key, value in REDUCTION_PARAMETERS.items() if value is not None}
tier = ExecutionTier(EXECUTION_TIER)
if tier is ExecutionTier.PREVIEW and not reductions:
    raise ValueError("preview execution must declare at least one reduction")
if tier is ExecutionTier.CANONICAL and reductions:
    raise ValueError(f"canonical execution cannot carry reductions: {sorted(reductions)}")
study = open_study(CASE_STUDY_ID, execution_tier=tier, workspace=WORKSPACE or None)
requests = {
    label: study.causal(
        method="dml",
        label=label,
        config_name="dml",
        execution_tier=tier,
        preview_reductions=reductions,
        overrides={},
        supersedes=causal_supersedes(
            study,
            SUPERSEDES_CAUSAL,
            label,
            labels=DECLARED_LABELS,
            execution_tier=tier.value,
        ),
    )
    for label in labels
}
resolutions = {label: request.resolve() for label, request in requests.items()}

# Seeded from the RESOLVED specification rather than from a notebook constant. The seed the
# estimate depends on is `dml.yaml`'s, and it is inside the identity hash - so a constant here
# that only reached `set_global_seeds` was a dial that turned without moving the number: change
# it and the identity is unchanged, the cached result fit under the configured seed is served
# back, and nothing says so. Seeding from the resolved value makes the seed this notebook
# announces the seed its results were actually fit under.
RANDOM_SEED = int(resolutions[labels[0]].spec["seed"])
if any(int(resolved.spec["seed"]) != RANDOM_SEED for resolved in resolutions.values()):
    raise RuntimeError(
        "the configured labels resolved to different seeds, so one global seed cannot describe "
        f"them: {sorted({int(r.spec['seed']) for r in resolutions.values()})}"
    )
set_global_seeds(RANDOM_SEED)

computations = {label: resolved.spec["computation"] for label, resolved in resolutions.items()}
computation = computations[labels[0]]

estimand = computation["estimand"]
print(f"Labels: {', '.join(labels)}")
print(f"Treatment: {estimand['treatment']}")
print(f"Confounders: {', '.join(estimand['confounders'])}")
print(f"Execution tier: {tier.value}")
print(f"Seed (from the resolved identity): {RANDOM_SEED}")
for horizon, values in computations.items():
    print(
        f"  {horizon}: outcome horizon {values['estimand']['outcome_horizon']}, "
        f"last admissible endpoint {values['estimand']['holdout_endpoint_cutoff']}"
    )

Labels: fwd_ret_1d, fwd_ret_5d, fwd_ret_21d
Treatment: mom_skip_recent
Confounders: vol_gk_21d, vol_gk_63d, zscore_21d
Execution tier: canonical
Seed (from the resolved identity): 42
  fwd_ret_1d: outcome horizon 1 days 00:00:00, last admissible endpoint 2023-12-29T00:00:00
  fwd_ret_5d: outcome horizon 5 days 00:00:00, last admissible endpoint 2023-12-22T00:00:00
  fwd_ret_21d: outcome horizon 21 days 00:00:00, last admissible endpoint 2023-11-30T00:00:00


## Inspect temporal controls

Cross-fitting groups observations by complete decision timestamps, and the embargo is at least the
outcome horizon.

Both of those are the panel version of a rule that is trivial in a cross-section. Grouping by
timestamp keeps every pair's observation for a session in the same fold, because splitting a
session across folds lets a nuisance model residualize one currency using another currency's
same-session outcome. The embargo drops the rows on either side of a fold boundary whose
outcomes overlap it, because a 21-session forward return computed on the last training session
is still being realized during the first validation sessions - the fold boundary separates the
decision times but not the outcomes those decisions are scored on.

The placebo permutes contiguous blocks within each currency pair rather than shuffling rows,
because two separate things make neighbouring rows dependent and a block has to span both.
Overlapping labels tie rows together across the outcome horizon. The treatment ties them together
across its own construction: `mom_skip_recent` is the return from `t-252` to `t-21`, so one value
covers 231 sessions and consecutive values overlap in 230 of them. `causal.treatment_window` in
`setup.yaml` declares 252, the oldest price any single value reads, which is the larger of the
three quantities in that expression and therefore spans the 231-session dependence whichever way
the arithmetic is read. The block takes the longer of that window and the label buffer.


<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 檢視時間控制措施

交叉擬合會按完整的決策時間戳記將觀測值分組，禁運期則至少與結果的預測期一樣長。

這兩項措施，是將在橫斷面中顯而易見的規則延伸至面板資料。依時間戳記分組，可確保同一交易時段每個貨幣對的觀測值都落在同一驗證折；若將同一時段拆到不同折，干擾模型便可能利用另一貨幣在同時段的結果，對某貨幣的觀測值取殘差。禁運期會移除驗證折邊界兩側那些結果觀測期跨越邊界的資料列，因為在最後一個訓練時段計算的 21 時段前瞻報酬率，在最初幾個驗證時段仍持續實現——驗證折邊界隔開的是決策時間，而非評分這些決策所用的結果。

安慰劑檢驗會在每個貨幣對內置換連續區塊，而非打亂個別資料列，因為有兩種不同因素使相鄰資料列相依，區塊必須涵蓋兩者。重疊的標籤會讓資料列在結果的預測期內相互連結。處置變數也因其自身的建構方式而使它們連結：`mom_skip_recent` 是從 `t-252` 到 `t-21` 的報酬率，因此單一數值涵蓋 231 個交易時段，而相鄰數值有 230 個時段重疊。`causal.treatment_window` 在 `setup.yaml` 中宣告為 252，也就是單一數值所讀取的最早價格；它是該運算式中三個量的最大者，因此無論如何理解算術關係，都能涵蓋 231 時段的相依性。區塊採用該視窗與標籤緩衝期兩者中較長者。
<!-- ML4T_ZH_END c0006-p000 -->

The table below prints the block the run used beside both scales it has to span, so the choice
can be checked rather than taken on trust. Until public #623 the block was sized from the label
buffer alone - 1, 5 and 21 sessions against a treatment whose dependence runs 231 - which is
close enough to an independent shuffle that it destroyed the dependence the permutation exists
to preserve, narrowing the placebo distribution and making `refutation_p` read stronger than the
evidence was.

<!-- ML4T_ZH_BEGIN c0006-p001 -->
下表將執行時所用的區塊大小，與它必須涵蓋的兩種尺度並列，讓這項選擇可供檢查，而不必照單全收。在公開版 #623 之前，區塊大小只根據標籤緩衝期設定——1、5 與 21 個交易時段，對照相依性長達 231 時段的處置變數——這已接近獨立打亂，因此破壞了置換檢驗本應保留的相依性，使安慰劑分布變窄，並讓 `refutation_p` 看起來比證據實際支持的更強。
<!-- ML4T_ZH_END c0006-p001 -->


In [4]:
pl.DataFrame(
    {
        "label": list(computations),
        "cross_fitting_folds": [c["cv"]["n_folds"] for c in computations.values()],
        "embargo_periods": [c["cv"]["embargo_periods"] for c in computations.values()],
        "fold_unit": [c["cv"]["fold_unit"] for c in computations.values()],
        # The two scales the block has to span, and which of them decided it. Read from the
        # resolved specification rather than from setup.yaml, so the table shows what the run
        # used rather than what the file asks for. `label_horizon_steps` is the outcome
        # horizon and is what the HAC bandwidth is sized by; `label_buffer_steps` is the CV
        # gap and is declared deliberately longer. They are different quantities and the
        # block is sized by neither on its own.
        "label_buffer_steps": [
            c["refutation"]["label_buffer_steps"] for c in computations.values()
        ],
        "label_horizon_steps": [
            c["refutation"]["label_horizon_steps"] for c in computations.values()
        ],
        "treatment_window_steps": [
            c["refutation"]["treatment_window_steps"] for c in computations.values()
        ],
        "placebo_method": [c["refutation"]["method"] for c in computations.values()],
        "placebo_block_size": [c["refutation"]["block_size"] for c in computations.values()],
        "block_size_basis": [c["refutation"]["block_size_basis"] for c in computations.values()],
        "analysis_rows": [c["analysis_population"]["n_rows"] for c in computations.values()],
        "analysis_timestamps": [
            c["analysis_population"]["n_timestamps"] for c in computations.values()
        ],
        "analysis_key_digest": [
            c["analysis_population"]["key_digest"] for c in computations.values()
        ],
    }
)

label,cross_fitting_folds,embargo_periods,fold_unit,label_buffer_steps,label_horizon_steps,treatment_window_steps,placebo_method,placebo_block_size,block_size_basis,analysis_rows,analysis_timestamps,analysis_key_digest
str,i64,i64,str,i64,i64,i64,str,i64,str,i64,i64,str
"""fwd_ret_1d""",5,1,"""complete_timestamp_panel""",1,1,252,"""within_symbol_contiguous_block…",252,"""treatment_window""",59540,2977,"""945fb65d5b24684a"""
"""fwd_ret_5d""",5,5,"""complete_timestamp_panel""",5,5,252,"""within_symbol_contiguous_block…",252,"""treatment_window""",59460,2973,"""8186088efd0836f9"""
"""fwd_ret_21d""",5,21,"""complete_timestamp_panel""",21,21,252,"""within_symbol_contiguous_block…",252,"""treatment_window""",59140,2957,"""3594630bc0b74a91"""


## Estimate or reload the causal result

Registration happens only after the fit returns a finite effect and HAC standard error. A missing
treatment, confounder, or valid analysis population fails before any causal row is written.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 估計或重新載入因果結果

只有在擬合回傳有限的效果估計值與 HAC 標準誤後，才會進行登記。若缺少處置變數、混雜變數或有效的分析樣本集合，就會在寫入任何因果資料列前失敗。
<!-- ML4T_ZH_END c0008-p000 -->


In [5]:
results = {}
for label, resolved in resolutions.items():
    result = resolved.run()
    if not result.complete:
        raise RuntimeError(f"the causal result for {label} is incomplete")
    # Compare the identity-bearing computation, not the whole specification. `provenance` records
    # the git commit of the run that registered the result, so a full-spec comparison fails on any
    # re-run made after any commit - it would assert that nothing had been committed since, which is
    # not a property of the causal estimate.
    if result.spec["computation"] != resolved.spec["computation"]:
        raise RuntimeError(f"the registered causal computation for {label} differs")
    reloaded = requests[label].resolve().run()
    if reloaded.hash != result.hash:
        raise RuntimeError(f"reloading the causal request for {label} changed its identity")
    results[label] = result

if len({result.hash for result in results.values()}) != len(labels):
    raise RuntimeError("two configured labels resolved to one causal identity")

for label, result in results.items():
    print(f"Registered causal identity, {label}: {result.hash}")
print("Effect estimates and their interpretation are reported in 12_model_analysis.")

Registered causal identity, fwd_ret_1d: 41a7a56a0118
Registered causal identity, fwd_ret_5d: 01fc2d8159c1
Registered causal identity, fwd_ret_21d: 84fb196c27f6
Effect estimates and their interpretation are reported in 12_model_analysis.


## Key takeaways

- Treatment, confounders, temporal geometry, and refutation settings are part of the estimand.
- Invalid specifications fail before registry mutation.
- Causal results remain distinct from predictive checkpoints and backtest candidates.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 重點摘要

- 處置、混雜變數、時間結構與反駁檢驗設定，都是估計目標的一部分。
- 無效的規格會在修改登錄系統前失敗。
- 因果結果與預測檢查點及回測候選者保持區別。
<!-- ML4T_ZH_END c0010-p000 -->
